In [1]:
from pathlib import Path
import pandas as pd
from entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError

api_key = Path('secrets/entsoe_api_key.txt').read_text(encoding='utf-8').strip()
client = EntsoePandasClient(api_key=api_key)

country_code = 'DE_LU'
tz = 'Europe/Berlin'
now = pd.Timestamp.now(tz=tz).floor('15min')
end = now.normalize() + pd.Timedelta(days=2)  # end of tomorrow

try:
    prices = client.query_day_ahead_prices(
        country_code,
        start=now,
        end=end,
        resolution='15T',
    )
    prices = prices[prices.index >= now]
    prices = prices[~prices.index.duplicated(keep='first')].sort_index()

    prices_df = prices.rename('price_eur_mwh').to_frame()
    prices_df.index.name = 'timestamp'
    prices_df = prices_df.reset_index()

    print(f'{country_code} available future rows: {len(prices_df)}')
    print(prices_df.head(10))
except NoMatchingDataError:
    prices_df = pd.DataFrame(columns=['timestamp', 'price_eur_mwh'])
    print('No future day-ahead prices are published yet.')


DE_LU available future rows: 147
                  timestamp  price_eur_mwh
0 2026-03-20 11:30:00+01:00          70.68
1 2026-03-20 11:45:00+01:00          62.56
2 2026-03-20 12:00:00+01:00          64.27
3 2026-03-20 12:15:00+01:00          61.40
4 2026-03-20 12:30:00+01:00          65.52
5 2026-03-20 12:45:00+01:00          59.11
6 2026-03-20 13:00:00+01:00          56.25
7 2026-03-20 13:15:00+01:00          54.03
8 2026-03-20 13:30:00+01:00          69.17
9 2026-03-20 13:45:00+01:00          65.63


In [6]:
prices_df["price_cent_kwh"] = prices_df['price_eur_mwh'] / 10
prices_df.tail(50)

,timestamp,price_eur_mwh,price_cent_kwh
97,2026-03-21 11:45:00+01:00,45.40,4.540
98,2026-03-21 12:00:00+01:00,51.97,5.197
99,2026-03-21 12:15:00+01:00,43.50,4.350
100,2026-03-21 12:30:00+01:00,39.10,3.910
101,2026-03-21 12:45:00+01:00,36.20,3.620
102,2026-03-21 13:00:00+01:00,50.06,5.006
103,2026-03-21 13:15:00+01:00,50.26,5.026
104,2026-03-21 13:30:00+01:00,50.36,5.036
105,2026-03-21 13:45:00+01:00,50.14,5.014
106,2026-03-21 14:00:00+01:00,61.21,6.121
